## Pilot and Main inter-annotator agreement

This notebook presents KA and ACC evaluations for each individual annotation phase, from the initial pilot stages through to the main annotation phases. The goal is to provide a comprehensive overview of the training process and the subsequent main annotation procedures in tabular form.

The tables for main+pilot, main only and pilot only agreement and accuracy are also available in the [Tables folder](../Tables/).


In [1]:
import pandas as pd
import krippendorff
import glob
import os

In [2]:
pilot_files = glob.glob('../Annotations/pilot*.csv')
main_files = glob.glob('../Annotations/main*.csv')

all_files = pilot_files + main_files

In [3]:
all_files

['../Annotations/pilot6.csv',
 '../Annotations/pilot4.csv',
 '../Annotations/pilot5.csv',
 '../Annotations/pilot1.csv',
 '../Annotations/pilot2.csv',
 '../Annotations/pilot3.csv',
 '../Annotations/main1.csv',
 '../Annotations/main2.csv']

In [4]:
import numpy as np

mapping_3class = {
    'Positive': 'Positive',
    'M_Positive': 'Positive',    
    'Negative': 'Negative',
    'M_Negative': 'Negative',
    'P_Neutral': 'Neutral',
    'N_Neutral': 'Neutral',
    }

results = []

for file in all_files:
    df = pd.read_csv(file)
    pilot_name = os.path.splitext(os.path.basename(file))[0]

    
    #Selecting columns with annotation tags (tag_tamara, tag_katja, tag_anze)
    tag_cols = [col for col in df.columns if col.startswith("tag_")]

    #---- 6-class KA calculation---#

    tags = df[tag_cols].copy()

    unique_tag6 = pd.unique(tags.values.ravel())
    unique_tag6 = [x for x in unique_tag6 if pd.notna(x)]

    tag_to_code6 = {label: idx for idx, label in enumerate(unique_tag6)}

    for col in tag_cols:
        df[col + '_6class'] = df[col].map(tag_to_code6)

    data_6class = df[[col + '_6class' for col in tag_cols]].to_numpy().T
    alpha_6class = krippendorff.alpha(reliability_data=data_6class, level_of_measurement='nominal')

    #---3-class KA calculation--#
    for col in tag_cols:
        df[col + '_3class'] = df[col].map(mapping_3class)

    unique_tag3 = pd.unique(df[[col + "_3class" for col in tag_cols]].values.ravel())
    unique_tag3 = [x for x in unique_tag3 if pd.notna(x)]
    
    tag_to_code3 = {label: idx for idx, label in enumerate(unique_tag3)}

    for col in tag_cols:
        df[col + '_3code'] = df[col + '_3class'].map(tag_to_code3)
    
    data_3class = df[[col + '_3code' for col in tag_cols]].to_numpy().T
    alpha_3class = krippendorff.alpha(reliability_data=data_3class, level_of_measurement="nominal")

    #----Percentage of agreed-upon instances---#
    def percent_agreement_complete(rows):
        if rows.isna().any():
            return np.nan
        return int(len(set(rows)) == 1)

    agreement_6 = (
        df[tag_cols]
        .apply(percent_agreement_complete, axis=1)
        .mean()
    )

    agreement_3 = (
        df[[col + '_3class' for col in tag_cols]]
        .apply(percent_agreement_complete, axis=1)
        .mean()
    )

    results.append({
        "phase": pilot_name,
        'KA_6class': round(alpha_6class, 4),
        'KA_3class': round(alpha_3class, 4),
        'ACC_6class': round(agreement_6 * 100, 1),
        'ACC_3class': round(agreement_3 * 100, 1)
    })

summary = pd.DataFrame(results)
summary = summary.sort_values(by="phase").reset_index(drop=True)

In [5]:
print(df[tag_cols].stack().unique())


['P_Neutral' 'Negative' 'N_Neutral' 'Positive' 'M_Positive' 'M_Negative']


In [6]:
summary.to_csv("../Tables/Pilot+Main_agreement.csv", encoding='utf-8', index=False)
summary

,phase,KA_6class,KA_3class,ACC_6class,ACC_3class
0,main1,0.4814,0.5082,62.6,72.5
1,main2,0.5430,0.5735,67.7,77.3
2,pilot1,0.3308,0.3655,29.4,38.2
3,pilot2,0.3316,0.3899,38.0,50.0
4,pilot3,0.3715,0.3870,34.0,44.0
5,pilot4,0.4754,0.5012,46.0,55.0
6,pilot5,0.3060,0.3377,48.0,58.0
7,pilot6,0.4363,0.3749,39.0,47.0


In [7]:
main = summary[summary['phase'].str.contains('main')]
main.to_csv("../Tables/Main_agreement.csv", encoding='utf-8', index=False)
main

,phase,KA_6class,KA_3class,ACC_6class,ACC_3class
0,main1,0.4814,0.5082,62.6,72.5
1,main2,0.5430,0.5735,67.7,77.3


In [8]:
pilot = summary[summary['phase'].str.contains('pilot')]
pilot.to_csv("../Tables/Pilot_agreement.csv", encoding='utf-8', index=False)
pilot

,phase,KA_6class,KA_3class,ACC_6class,ACC_3class
2,pilot1,0.3308,0.3655,29.4,38.2
3,pilot2,0.3316,0.3899,38.0,50.0
4,pilot3,0.3715,0.3870,34.0,44.0
5,pilot4,0.4754,0.5012,46.0,55.0
6,pilot5,0.3060,0.3377,48.0,58.0
7,pilot6,0.4363,0.3749,39.0,47.0
